# 05. Stable Diffusion VAE 解剖

使用 SD 1.5 VAE 将真实图像压缩到 4-channel latent，再解码回 RGB。使用 posterior mode 而不是随机 sample，保证重跑时重构一致。

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from diffusers import AutoencoderKL

MODEL_ID = os.environ.get('P3_MODEL_ID', 'stable-diffusion-v1-5/stable-diffusion-v1-5')
MODEL_REVISION = os.environ.get('P3_MODEL_REVISION', '451f4fe16113bff5a5d2269ed5ad43b0592e9a14')
OUTPUT_DIR = Path(os.environ.get('P3_OUTPUT_DIR', 'outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if device.type == 'cuda' else torch.float32
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae', revision=MODEL_REVISION).to(device, dtype=dtype).eval()
vae.requires_grad_(False)
print('device:', device, 'dtype:', dtype, 'scaling_factor:', vae.config.scaling_factor)

In [ ]:
candidate = os.environ.get('P3_VAE_IMAGE', '.local/datasets/project3_vangogh/vangogh_00.jpg')
if not Path(candidate).exists():
    candidate = 'input.png'
image = Image.open(candidate).convert('RGB')
preprocess = transforms.Compose([transforms.Resize(512, interpolation=transforms.InterpolationMode.BILINEAR), transforms.CenterCrop(512), transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
pixel_values = preprocess(image).unsqueeze(0).to(device, dtype=dtype)
with torch.no_grad():
    posterior = vae.encode(pixel_values).latent_dist
    latent = posterior.mode() * vae.config.scaling_factor
    reconstruction = vae.decode(latent / vae.config.scaling_factor).sample
print('input:', tuple(pixel_values.shape), 'latent:', tuple(latent.shape), 'reconstruction:', tuple(reconstruction.shape))
reconstruction_01 = (reconstruction / 2 + 0.5).clamp(0, 1)
original_01 = (pixel_values / 2 + 0.5).clamp(0, 1)

In [ ]:
# 每个 channel 单独 min-max 显示，避免某个 channel 的尺度掩盖其它 channel。
z = latent[0].float().cpu().numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for channel, ax in enumerate(axes):
    value = z[channel]
    normalized = (value - value.min()) / (value.max() - value.min() + 1e-8)
    ax.imshow(normalized, cmap='viridis')
    ax.set_title(f'latent channel {channel}')
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'vae_channels.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
original_np = original_01[0].permute(1, 2, 0).float().cpu().numpy()
recon_np = reconstruction_01[0].permute(1, 2, 0).float().cpu().numpy()
mse = float(np.mean((original_np - recon_np) ** 2))
psnr = float(-10 * np.log10(max(mse, 1e-12)))
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_np); axes[0].set_title('original'); axes[0].axis('off')
axes[1].imshow(recon_np); axes[1].set_title(f'reconstruction | MSE={mse:.5f}, PSNR={psnr:.2f} dB'); axes[1].axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'vae_reconstruction.png', dpi=150, bbox_inches='tight'); plt.show()
# 放大右下角，便于观察纹理、细小文字和边缘。
crop = slice(320, 512)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original_np[crop, crop]); axes[0].set_title('original crop'); axes[0].axis('off')
axes[1].imshow(recon_np[crop, crop]); axes[1].set_title('reconstruction crop'); axes[1].axis('off')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'vae_detail_crop.png', dpi=150, bbox_inches='tight'); plt.show()
metrics = {'input_path': str(Path(candidate)), 'input_shape': list(pixel_values.shape), 'latent_shape': list(latent.shape), 'reconstruction_shape': list(reconstruction.shape), 'mse_01': mse, 'psnr_db': psnr, 'latent_channel_mean': z.mean(axis=(1,2)).tolist(), 'latent_channel_std': z.std(axis=(1,2)).tolist()}
(OUTPUT_DIR / 'vae_metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))